# Web Security Scanner — Jupyter

Run the same security-header / CSP / JS-vulnerability scanner from `web_security_scanner.py` interactively, with results displayed inline.

This notebook **imports** the scanner module — there is no duplicated logic. Any change to `web_security_scanner.py` is picked up after a kernel restart.

## 1. Install dependencies

Uncomment the lines you need the first time you run this notebook.

In [1]:
# %pip install -r requirements.txt pandas
# Optional — for --deep mode (captures JS-injected resources):
# %pip install playwright
# !playwright install chromium
# Optional — for retire.js JS vulnerability scanning:
# !npm install -g retire

## 2. Imports

In [2]:
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Make sure the notebook can import the scanner module from this directory
sys.path.insert(0, str(Path.cwd()))

import web_security_scanner as wss
import pandas as pd

print(f"Scanner module loaded from: {wss.__file__}")

Scanner module loaded from: /opt/enum/security-header/web_security_scanner.py


## 3. Configure URLs and options

Edit the list below — no `input.txt` file needed.

In [3]:
URLS = [
    "https://example.com",
    # "https://your-target.com",
]

OPTIONS = {
    "timeout": 15,        # HTTP request timeout (seconds)
    "threads": 3,         # concurrent scans
    "use_retire": False,  # True if retire.js is installed (npm install -g retire)
    "skip_csp": False,    # True to skip CSP header checks entirely
    "gen_csp": True,      # auto-generate a strict CSP for every URL
    "deep": False,        # use Playwright to capture JS-injected sources (slower, more accurate)
}

## 4. Run the scan

In [4]:
session = wss.get_session()
results = []

print(f"Scanning {len(URLS)} URL(s) with {OPTIONS['threads']} thread(s)…")

with ThreadPoolExecutor(max_workers=OPTIONS["threads"]) as executor:
    future_to_url = {
        executor.submit(
            wss.scan_url, url, session,
            OPTIONS["use_retire"], OPTIONS["timeout"], OPTIONS["skip_csp"],
            OPTIONS["gen_csp"], OPTIONS["deep"],
        ): url for url in URLS
    }
    for i, future in enumerate(as_completed(future_to_url), 1):
        url = future_to_url[future]
        try:
            result = future.result()
            results.append(result)
            crit, high, med, low = wss.count_severities(result)
            print(f"  [{i}/{len(URLS)}] {result['url']} (HTTP {result['status']}) "
                  f"— {crit} CRIT / {high} HIGH / {med} MED / {low} LOW")
        except Exception as e:
            print(f"  [{i}/{len(URLS)}] {url} — SCAN ERROR: {e}")

print(f"\nDone. {len(results)} result(s).")

Scanning 1 URL(s) with 3 thread(s)…


  [1/1] https://example.com (HTTP 200) — 0 CRIT / 6 HIGH / 5 MED / 1 LOW

Done. 1 result(s).


## 5. Summary table

In [5]:
rows = []
for r in sorted(results, key=lambda x: x["url"]):
    crit, high, med, low = wss.count_severities(r)
    rows.append({
        "URL": r["url"],
        "HTTP": r["status"],
        "Issues": "YES" if wss.has_issues(r) else "NO",
        "CRIT": crit, "HIGH": high, "MED": med, "LOW": low,
        "Has CSP suggestion": "yes" if r.get("csp_suggestion") else "no",
    })
summary_df = pd.DataFrame(rows)
summary_df

,URL,HTTP,Issues,CRIT,HIGH,MED,LOW,Has CSP suggestion
0,https://example.com,200,YES,0,6,5,1,yes


## 6. Findings detail per URL

In [6]:
for r in sorted(results, key=lambda x: x["url"]):
    print("=" * 70)
    print(r["url"])
    print("=" * 70)
    print(wss.format_findings(r))
    print()

https://example.com
=== HTTP Status: 200 ===

--- Security Headers ---
  [HIGH] Strict-Transport-Security: MISSING - HSTS - Forces HTTPS connections. Recommended: max-age=31536000; includeSubDomains; preload
  [HIGH] Content-Security-Policy: MISSING - CSP - Mitigates XSS and injection attacks. Recommended: default-src 'self'; script-src 'self'; object-src 'none'
  [HIGH] X-Content-Type-Options: MISSING - Prevents MIME-type sniffing. Recommended: nosniff
  [HIGH] X-Frame-Options: MISSING - Prevents clickjacking via framing. Recommended: DENY or SAMEORIGIN
  [HIGH] Referrer-Policy: MISSING - Controls referrer information sent with requests. Recommended: strict-origin-when-cross-origin or no-referrer
  [HIGH] Permissions-Policy: MISSING - Controls browser feature access (camera, mic, geolocation, etc.). Recommended: geolocation=(), camera=(), microphone=()
  [MEDIUM] X-XSS-Protection: MISSING - Legacy XSS filter (deprecated, but absence noted). Recommended: 0 (disable) or absent with stro

## 7. Generated CSP per URL

Up to three policies per URL, in order of preference:

1. **Strict** — no `'unsafe-inline'`. Use after refactoring inline content.
2. **Hash-based** — `'sha256-XYZ='` tokens for detected inline blocks. **Bitsight-friendly** alternative to `'unsafe-inline'`. Most accurate with `OPTIONS['deep']=True` (captures JS-injected content).
3. **Practical** — includes `'unsafe-inline'`. Works immediately but penalised by security-ratings services.

Hashes are stable per page version — update when the page is redeployed.

In [7]:
def _print_csp(label, csp):
    print(f"# {label}")
    print("Content-Security-Policy:")
    for directive in csp.split("; "):
        print(f"  {directive};")

for r in sorted(results, key=lambda x: x["url"]):
    strict = r.get("csp_suggestion")
    practical = r.get("csp_suggestion_practical")
    hashed = r.get("csp_suggestion_hashed")
    if not strict:
        continue
    print(f"--- {r['url']} ---\n")
    _print_csp("Strict (recommended — requires refactoring inline content)", strict)
    if hashed:
        print()
        _print_csp("Hash-based (Bitsight-friendly — no unsafe-inline)", hashed)
    if practical:
        print()
        _print_csp("Practical (works immediately but penalised by Bitsight)", practical)
    notes = r.get("csp_notes") or []
    if notes:
        print("\nWarnings:")
        for n in notes:
            print(f"  - {n}")
    print()


--- https://example.com ---

# Strict (recommended — requires refactoring inline content)
Content-Security-Policy:
  default-src 'self';
  script-src 'self';
  style-src 'self';
  img-src 'self' data:;
  font-src 'self';
  connect-src 'self';
  object-src 'none';
  form-action 'self';
  base-uri 'self';
  frame-ancestors 'none';
  upgrade-insecure-requests;

# Hash-based (Bitsight-friendly — no unsafe-inline)
Content-Security-Policy:
  default-src 'self';
  script-src 'self';
  style-src 'self' 'sha256-bPz1p+hPRvUB6/coXO5smlJ00u0kwHOGz+NGVcCSKeM=';
  img-src 'self' data:;
  font-src 'self';
  connect-src 'self';
  object-src 'none';
  form-action 'self';
  base-uri 'self';
  frame-ancestors 'none';
  upgrade-insecure-requests;

# Practical (works immediately but penalised by Bitsight)
Content-Security-Policy:
  default-src 'self';
  script-src 'self';
  style-src 'self' 'unsafe-inline';
  img-src 'self' data:;
  font-src 'self';
  connect-src 'self';
  object-src 'none';
  form-action 

## 8. Export results to CSV (optional)

In [8]:
output_path = "results.csv"
wss.write_csv(output_path, results)
print(f"Wrote {len(results)} row(s) to {output_path}")

Wrote 1 row(s) to results.csv


## 9. Server config snippets (optional)

The scanner can emit ready-to-paste Apache and Nginx blocks for the same recommendations — useful if you want to apply changes immediately.

In [9]:
for r in sorted(results, key=lambda x: x["url"]):
    print("=" * 70)
    print(r["url"])
    print("=" * 70)
    print("\n--- Nginx ---\n")
    print(wss.build_nginx_config(r))
    print("\n--- Apache ---\n")
    print(wss.build_apache_config(r))
    print()

https://example.com

--- Nginx ---

# Nginx — place inside the relevant server {} block,
# or include via /etc/nginx/conf.d/security-headers.conf.
add_header Strict-Transport-Security "max-age=31536000; includeSubDomains; preload" always;
add_header Content-Security-Policy "default-src 'self'; script-src 'self'; style-src 'self'; img-src 'self' data:; font-src 'self'; connect-src 'self'; object-src 'none'; form-action 'self'; base-uri 'self'; frame-ancestors 'none'; upgrade-insecure-requests" always;
add_header X-Content-Type-Options "nosniff" always;
add_header X-Frame-Options "DENY" always;
add_header Referrer-Policy "strict-origin-when-cross-origin" always;
add_header Permissions-Policy "geolocation=(), camera=(), microphone=()" always;
add_header X-XSS-Protection "0" always;
add_header Cross-Origin-Opener-Policy "same-origin" always;
add_header Cross-Origin-Resource-Policy "same-origin" always;
add_header Cross-Origin-Embedder-Policy "require-corp" always;
add_header Cache-Control 

## 10. Nginx nonce-injection template

For pages with inline content you can't refactor (third-party apps), the scanner also emits a ready-to-paste nginx config that uses per-request nonces + `'strict-dynamic'`. This is the **Bitsight-friendliest** way to handle MinIO-style React-bundle pages.

In [10]:
for r in sorted(results, key=lambda x: x["url"]):
    template = wss.build_nginx_nonce_template(r)
    if not template:
        continue
    print("=" * 70)
    print(r["url"])
    print("=" * 70)
    print(template)
    print()


https://example.com
# Nginx + per-request CSP nonce — Bitsight/SecurityScorecard-friendly
# alternative to 'unsafe-inline'. Generated for: https://example.com
#
# Requirements (one of):
#   - nginx-extras package (apt install nginx-extras) for set_secure_random_alphanum
#   - OR OpenResty / nginx + lua-nginx-module
#
# Place inside the relevant 'location' or 'server' block.

# 1. Fresh 32-char nonce for every request.
set_secure_random_alphanum $cspNonce 32;

# 2. Disable upstream compression so sub_filter can rewrite the response body.
proxy_set_header Accept-Encoding "";

# 3. Inject nonce into every <script> and <style> tag.
sub_filter_once off;
sub_filter_types text/html;
sub_filter '<script'  '<script nonce="$cspNonce"';
sub_filter '<style'   '<style nonce="$cspNonce"';

# 4. CSP using 'nonce-...' + 'strict-dynamic'. The nonce-allowed root script
#    can then load other scripts transitively without origin allowlisting.
add_header Content-Security-Policy "\
default-src 'self'; \
s